# 006-segmentation-ceiling — analysis

# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.track import experiment_dir, load_runs

# Params

In [ ]:
EXP = "006-segmentation-ceiling"
BASELINE = "006.1"
GT_QUERY = "006.3"
STRIDE_RUNS = {1: "006.3", 2: "006.6", 4: "006.7", 8: "006.8", 16: "006.9"}
BOUNDARY_RUNS = {0: "006.3", 1: "006.10", 2: "006.11", 4: "006.12", 8: "006.13"}
RULE_1_GAIN_M = 2.0
RULE_2_SHARE = 0.5
RULE_2_RUNS = ["006.8", "006.11"]
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 15.0
N_BOOTSTRAP = 10_000
CI_PCT = 95
SEED = 42

# Data

In [ ]:
runs = {r["run"]: r for r in load_runs(EXP) if r["status"] == "ok"}
records = {run: pd.read_csv(experiment_dir(EXP) / "artifacts" / f"records-{run}.csv") for run in runs}
list(runs)

In [ ]:
def describe(run: dict) -> str:
    c = run["config"]
    query = c["query_source"] if not c.get("gt_classes") or c["gt_classes"] == "all" else f"model + GT {','.join(c['gt_classes'])}"
    damage = "" if c["damage"] == "none" else f", {c['damage']} {c['damage_px']}"
    return f"query {query}{damage} | map {c['map_source']} | labels {c['label_source']}"

# Run table

In [ ]:
table = pd.DataFrame({run: {"setup": describe(r),
                            **{f"@{n:g} m": r["metrics"][f"mean_position_error_m_at_{n:g}m"] for n in NOISE_M},
                            "median @15": r["metrics"]["median_position_error_m"],
                            "kept prior @15": r["metrics"]["kept_prior_share"],
                            "bushcha @15": r["metrics"]["mean_position_error_m_bushcha"],
                            "birky @15": r["metrics"]["mean_position_error_m_birky"]}
                      for run, r in runs.items()}).T
table.round(2)

In [ ]:
query_shares = pd.DataFrame({run: {k.removeprefix("query_share_"): v for k, v in r["metrics"].items() if k.startswith("query_share_")}
                             for run, r in runs.items()}).T
query_shares.round(3)

# Paired bootstrap

In [ ]:
def per_frame(run: str, noise_m: float) -> pd.Series:
    r = records[run]
    return r[r["noise_m"] == noise_m].groupby("frame")["error_m"].mean()

In [ ]:
def paired_diff(run: str, reference: str, noise_m: float) -> dict[str, float]:
    """Mean of `run` - `reference`, with a frame-bootstrap CI and the number of frames where `run` is better."""
    diff = (per_frame(run, noise_m) - per_frame(reference, noise_m)).to_numpy()
    rng = np.random.default_rng(SEED)
    boot = diff[rng.integers(0, len(diff), (N_BOOTSTRAP, len(diff)))].mean(1)
    lo, hi = np.percentile(boot, [(100 - CI_PCT) / 2, 100 - (100 - CI_PCT) / 2])
    return {"diff_m": diff.mean(), "ci_lo": lo, "ci_hi": hi, "frames_better": int((diff < 0).sum()), "frames": len(diff)}

In [ ]:
vs_baseline = pd.DataFrame({(run, n): paired_diff(run, BASELINE, n) for run in runs if run != BASELINE for n in NOISE_M}).T
vs_baseline.round(2)

In [ ]:
vs_gt_query = pd.DataFrame({(run, n): paired_diff(run, GT_QUERY, n)
                            for run in [*STRIDE_RUNS.values(), *BOUNDARY_RUNS.values()] if run != GT_QUERY and run in runs
                            for n in NOISE_M}).T
vs_gt_query.round(2)

# Rules

In [ ]:
rule_1 = paired_diff(GT_QUERY, BASELINE, PRIMARY_NOISE_M)
rule_1_pass = rule_1["diff_m"] <= -RULE_1_GAIN_M and rule_1["ci_hi"] < 0
{"006.3 - 006.1, m": round(rule_1["diff_m"], 2), "CI": (round(rule_1["ci_lo"], 2), round(rule_1["ci_hi"], 2)),
 "rule 1: segmentation worth improving": bool(rule_1_pass)}

In [ ]:
def err(run: str) -> float:
    return runs[run]["metrics"]["mean_position_error_m"]

In [ ]:
gap_m = err(BASELINE) - err(GT_QUERY)
rule_2_bar = err(GT_QUERY) + RULE_2_SHARE * gap_m
{"G, m": round(gap_m, 2), "bar, m": round(rule_2_bar, 2), **{run: round(err(run), 2) for run in RULE_2_RUNS},
 "rule 2: edges are the cause": bool(rule_1_pass and any(err(run) >= rule_2_bar for run in RULE_2_RUNS)),
 "rule 2 evaluated": bool(rule_1_pass)}

# Plots

In [ ]:
fig_curves, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (name, curve, unit) in zip(axes, [("stride", STRIDE_RUNS, "s"), ("boundary damage", BOUNDARY_RUNS, "r, px")]):
    for noise_m in NOISE_M:
        xs = [x for x, run in curve.items() if run in runs]
        line, = ax.plot(xs, [runs[curve[x]]["metrics"][f"mean_position_error_m_at_{noise_m:g}m"] for x in xs], "o-", label=f"GT query, {noise_m:g} m")
        ax.axhline(runs[BASELINE]["metrics"][f"mean_position_error_m_at_{noise_m:g}m"], color=line.get_color(), ls="--", lw=1)
    ax.set_xscale("symlog", base=2, linthresh=1); ax.set_xticks(list(curve)); ax.set_xticklabels(list(curve))
    ax.set_xlabel(f"{name}, {unit}"); ax.set_title(f"GT query, {name} (dashed: 006.1)"); ax.grid(alpha=.3)
axes[0].set_ylabel("mean position error, m"); axes[0].legend(fontsize=8)
fig_curves.tight_layout()
fig_curves.savefig(experiment_dir(EXP) / "figures" / "damage-curves.png", dpi=110)

In [ ]:
fig_bars, ax = plt.subplots(figsize=(9, 3.8))
primary = table["@15 m"].astype(float)
ax.barh([f"{run}  {table.loc[run, 'setup']}" for run in primary.index], primary.to_numpy())
ax.axvline(err(BASELINE), color="k", ls="--", lw=1, label="006.1 baseline")
ax.axvline(err(BASELINE) - RULE_1_GAIN_M, color="r", ls=":", lw=1, label="rule 1 bar")
ax.invert_yaxis(); ax.set_xlabel(f"mean position error @ {PRIMARY_NOISE_M:g} m, m"); ax.legend(fontsize=8)
fig_bars.tight_layout()
fig_bars.savefig(experiment_dir(EXP) / "figures" / "runs-at-15m.png", dpi=110)